In [1]:
# import pandas as pd
# import yfinance as yf
# start_date = '2015-02-01'
# end_date = '2025-10-31'
# qqq = yf.download('QQQ', start=start_date, end=end_date, auto_adjust=False)['Close'].resample('ME').last().reset_index()
# qqq.to_csv('data/qqq_benchmark.csv', index=False)
# qqq

In [2]:
"""
Fixed Hierarchical Agent Training System - ENHANCED
==================================================

Fixes Applied:
1. Walk-forward validation (no regime mismatch)
2. Super Agent sees FULL information (not just compressed weights)
3. SAC algorithm for Super Agent (better exploration)
4. Multi-window ensemble (not just best window)
5. Optimal hyperparameters from audit

"""

import numpy as np
import pandas as pd
import gymnasium
from gymnasium import spaces
from stable_baselines3 import PPO, SAC
from stable_baselines3.common.callbacks import BaseCallback
from sklearn.preprocessing import StandardScaler
import warnings
import json
import os
from typing import Dict, List, Tuple
warnings.filterwarnings('ignore')

from environments import SuperAgentEnv
from environments import MetaAgentEnv
from utile import EarlyStoppingCallback
from walk_forward_validation import WalkForwardValidator
from walk_forward_validation import train_super_agent_walk_forward_sac, train_meta_agent_walk_forward
from production_agent_wrapper_frozen import ProductionAgentWrapper

# ============================================================================
# OPTIMAL CONFIGURATION
# ============================================================================

OPTIMAL_CONFIG = {
    # Data Splitting
    'use_walk_forward': True,
    'walk_forward_windows': 5,
    'min_train_months': 60,
    
    # Fallback
    'train_ratio': 0.70,
    'val_ratio': 0.15,
    'test_ratio': 0.15,
    
    # SUPER AGENT (Now with SAC)
    'super_learning_rate': 0.001,      # SAC typically needs higher LR
    'super_timesteps': 100_000,
    'super_network': [256, 128],        # Larger network for more features
    'super_batch_size': 256,
    'super_buffer_size': 50_000,        # SAC replay buffer
    'super_tau': 0.005,                 # SAC soft update
    'super_gamma': 0.98,
    'super_ent_coef': 'auto',           # SAC auto entropy tuning
    'super_eval_freq': 2_000,
    'super_patience': 10,
    'super_min_delta': 0.01,
    
    # META AGENT (Stays PPO)
    'meta_learning_rate': 0.0001,
    'meta_timesteps': 50_000,
    'meta_network': [64, 32],
    'meta_batch_size': 256,
    'meta_n_steps': 2048,
    'meta_n_epochs': 3,
    'meta_gamma': 0.98,
    'meta_ent_coef': 0.005,
    'meta_max_grad_norm': 0.5,
    'meta_gae_lambda': 0.95,
    'meta_eval_freq': 2_000,
    'meta_patience': 10,
    'meta_min_delta': 0.005,
    
    # Risk Constraints
    'max_position': 0.30,
    'min_position': 0.05,
    'max_turnover': 0.50,
    'min_market_exposure': 0.90,
    'transaction_cost': 0.002,
    
    # Reward Function
    'alpha_returns': 3.0,
    'alpha_mdd': 1.0,
    'alpha_vol': 0.5,
    'alpha_concentration': 0.3,
    'exploration_bias': 0.01,
    'constraint_penalty': 50.0,
    
    # Training Stability
    'seed': 42,
    'min_training_episodes': 5,
    'verbose': 1,
}



In [3]:

print("ENHANCED HIERARCHICAL SYSTEM")
print("="*70)
print("\nEnhancements:")
print("  ✨ Super sees FULL features (base weights + technical + sentiment)")
print("  ✨ Super uses SAC algorithm (better exploration)")
print("  ✨ Multi-window ensemble (all windows, not just best)")
print("\nArchitecture:")
print("  Base:  4 agents (tech_PPO, tech_SAC, sent_PPO, sent_SAC)")
print("  Super: SAC with enhanced observations")
print("  Meta:  PPO adjusting for regimes")

# Load data
print("\n[1/6] Loading data...")
price_data = pd.read_csv('data/price_data.csv', index_col=0, parse_dates=True)
technical_features = pd.read_csv('data/technical_features.csv', index_col=0, parse_dates=True)
sentiment_features = pd.read_csv('data/sentiment_features.csv', index_col=0, parse_dates=True)
regime = pd.read_csv('data/regime_indicators.csv', index_col=0, parse_dates=True)

qqq_benchmark = pd.read_csv('data/qqq_benchmark.csv', index_col=0, parse_dates=True)
print(f"   ✓ QQQ benchmark loaded")

print(f"\n   Trading data: {price_data.shape}")
print(f"   Assets: {list(price_data.columns)}")

# Split data
print("\n[2/6] Creating walk-forward splits...")

validator = WalkForwardValidator(
    n_windows=OPTIMAL_CONFIG['walk_forward_windows'],
    min_train_months=OPTIMAL_CONFIG['min_train_months']
)
data_splits, test_data = validator.split_data_walk_forward(
    price_data, technical_features, sentiment_features, regime
)

test_prices, test_tech, test_sent, test_regime = test_data

# QQQ baseline
print("\n[3/6] Calculating QQQ baseline...")
if qqq_benchmark is not None:
    test_qqq = qqq_benchmark.loc[test_prices.index]
    qqq_returns = test_qqq['QQQ'].pct_change().dropna()
    qqq_total_return = (test_qqq['QQQ'].iloc[-1] / test_qqq['QQQ'].iloc[0]) - 1
    qqq_total_return -= 2 * OPTIMAL_CONFIG['transaction_cost']
    
    qqq_sharpe = 0.0
    if len(qqq_returns) > 0 and qqq_returns.std() > 0:
        qqq_sharpe = qqq_returns.mean() / qqq_returns.std() * np.sqrt(12)
    
    cumulative = (1 + qqq_returns).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    qqq_max_dd = abs(drawdown.min())
    
    qqq_volatility = qqq_returns.std() * np.sqrt(12)
    qqq_win_rate = (qqq_returns > 0).sum() / len(qqq_returns)
    
    qqq_metrics = {
        'strategy': 'QQQ Buy & Hold',
        'total_return': qqq_total_return,
        'sharpe_ratio': qqq_sharpe,
        'max_drawdown': qqq_max_dd,
        'volatility': qqq_volatility,
        'win_rate': qqq_win_rate,
        'final_value': 100_000 * (1 + qqq_total_return)
    }
    
    print(f"\n   QQQ Baseline: Sharpe={qqq_sharpe:.3f}, Return={qqq_total_return*100:.1f}%")
else:
    qqq_metrics = None

# Train Enhanced Super agent
print("\n[4/6] Training Enhanced Super agent...")
super_ensemble, super_val_sharpes = train_super_agent_walk_forward_sac(
    data_splits, OPTIMAL_CONFIG
)

# Save ensemble
os.makedirs('models', exist_ok=True)
super_ensemble.save('models/super_agent_enhanced_ensemble')
print(f"\n   ✓ Saved: models/super_agent_enhanced_ensemble")

# Train Meta agent
print("\n[5/6] Training Meta agent...")
meta_ensemble, meta_val_sharpes = train_meta_agent_walk_forward(
    data_splits, super_ensemble, OPTIMAL_CONFIG
)

# Save ensemble
meta_ensemble.save('models/meta_agent_ensemble')
print(f"\n   ✓ Saved: models/meta_agent_ensemble")

# Final evaluation
print("\n[6/6] Final evaluation on holdout test set...")

n_assets = len(test_prices.columns)

# Load base agents
print(f"\n   Loading base agents...")
tech_ppo_model = PPO.load('models/best_technical_PPO.zip')
tech_sac_model = SAC.load('models/best_technical_SAC.zip')
sent_ppo_model = PPO.load('models/best_sentiment_PPO.zip')
sent_sac_model = SAC.load('models/best_sentiment_SAC.zip')

base_agents_test = {
    'tech_PPO': ProductionAgentWrapper(tech_ppo_model, test_tech, n_assets, 'tech_PPO'),
    'tech_SAC': ProductionAgentWrapper(tech_sac_model, test_tech, n_assets, 'tech_SAC'),
    'sent_PPO': ProductionAgentWrapper(sent_ppo_model, test_sent, n_assets, 'sent_PPO'),
    'sent_SAC': ProductionAgentWrapper(sent_sac_model, test_sent, n_assets, 'sent_SAC'),
}

# Evaluate Enhanced Super
print(f"\n   Evaluating Enhanced Super ensemble...")
super_test_env = SuperAgentEnv(
    test_prices, base_agents_test, test_tech, test_sent, **OPTIMAL_CONFIG
)
obs, _ = super_test_env.reset()
done = False
while not done:
    action, _ = super_ensemble.predict(obs, deterministic=True)
    obs, _, done, _, _ = super_test_env.step(action)
super_metrics = super_test_env.get_portfolio_metrics()
super_metrics['strategy'] = 'Enhanced Super (SAC+Full)'

# Evaluate Meta
print(f"   Evaluating Meta ensemble...")
super_test_wrap = ProductionAgentWrapper(super_ensemble, super_test_env, n_assets, 'super')
meta_test_env = MetaAgentEnv(test_prices, super_test_wrap, test_regime, **OPTIMAL_CONFIG)
obs, _ = meta_test_env.reset()
done = False
while not done:
    action, _ = meta_ensemble.predict(obs, deterministic=True)
    obs, _, done, _, _ = meta_test_env.step(action)
meta_metrics = meta_test_env.get_portfolio_metrics()
meta_metrics['strategy'] = 'Meta Agent (Ensemble)'

# Best base agent
print(f"   Evaluating best base agent...")
best_base_env = SuperAgentEnv(
    test_prices, {'sent_PPO': base_agents_test['sent_PPO']}, 
    test_tech, test_sent, **OPTIMAL_CONFIG
)
obs, _ = best_base_env.reset()
done = False
while not done:
    action = base_agents_test['sent_PPO'].weights
    obs, _, done, _, _ = best_base_env.step(action)
best_base_metrics = best_base_env.get_portfolio_metrics()
best_base_metrics['strategy'] = 'Best Base (sent_PPO)'

# Results
print("\n" + "="*70)
print("RESULTS")
print("="*70)

print(f"\nEnhanced Super Agent:")
print(f"  Avg Val Sharpe:  {np.mean(super_val_sharpes):.3f} (±{np.std(super_val_sharpes):.3f})")
print(f"  Test Sharpe:     {super_metrics['sharpe_ratio']:.3f}")
print(f"  Val/Test Ratio:  {super_metrics['sharpe_ratio']/np.mean(super_val_sharpes):.2f}x")

print(f"\nMeta Agent:")
print(f"  Avg Val Sharpe:  {np.mean(meta_val_sharpes):.3f} (±{np.std(meta_val_sharpes):.3f})")
print(f"  Test Sharpe:     {meta_metrics['sharpe_ratio']:.3f}")
print(f"  Val/Test Ratio:  {meta_metrics['sharpe_ratio']/np.mean(meta_val_sharpes):.2f}x")

# Comparison table
print("\n" + "="*70)
print("FINAL COMPARISON (Test Set)")
print("="*70)

all_strategies = [best_base_metrics, super_metrics, meta_metrics]
if qqq_metrics:
    all_strategies.insert(0, qqq_metrics)

print(f"\n{'Strategy':<30s} {'Sharpe':>10s} {'Return':>10s} {'Max DD':>10s} {'Win Rate':>10s}")
print("-"*70)

for metrics in all_strategies:
    strategy_name = metrics.get('strategy', 'Unknown')
    print(f"{strategy_name:<30s} "
            f"{metrics['sharpe_ratio']:>10.3f} "
            f"{metrics['total_return']*100:>9.1f}% "
            f"{metrics['max_drawdown']*100:>9.1f}% "
            f"{metrics['win_rate']*100:>9.1f}%")

# Performance vs QQQ
if qqq_metrics:
    print("\n" + "="*70)
    print("PERFORMANCE vs QQQ")
    print("="*70)
    
    baseline_sharpe = qqq_metrics['sharpe_ratio']
    
    strategies_to_compare = [
        ('Best Base Agent', best_base_metrics),
        ('Enhanced Super Agent', super_metrics),
        ('Meta Agent', meta_metrics)
    ]
    
    print(f"\n{'Strategy':<30s} {'vs QQQ':>12s} {'Assessment':>20s}")
    print("-"*70)
    
    for name, metrics in strategies_to_compare:
        improvement = ((metrics['sharpe_ratio'] - baseline_sharpe) / baseline_sharpe * 100) if baseline_sharpe > 0 else 0
        
        if improvement > 20:
            assessment = "✅ Excellent"
        elif improvement > 5:
            assessment = "✓ Good"
        elif improvement > -5:
            assessment = "≈ Comparable"
        else:
            assessment = "⚠️  Underperforms"
        
        print(f"{name:<30s} {improvement:>11.1f}% {assessment:>20s}")

# Diagnostics
print("\n" + "="*70)
print("DIAGNOSTICS")
print("="*70)

print(f"\n1. Window Consistency:")
print(f"   Enhanced Super: {[f'{s:.3f}' for s in super_val_sharpes]}")
print(f"   Meta:           {[f'{s:.3f}' for s in meta_val_sharpes]}")
print(f"   Super variance: {np.var(super_val_sharpes):.4f}")
print(f"   Meta variance:  {np.var(meta_val_sharpes):.4f}")

print(f"\n2. Hierarchy Value:")
print(f"   Base → Enhanced Super: {best_base_metrics['sharpe_ratio']:.3f} → {super_metrics['sharpe_ratio']:.3f} "
        f"({(super_metrics['sharpe_ratio']/best_base_metrics['sharpe_ratio']-1)*100:+.1f}%)")
print(f"   Enhanced Super → Meta: {super_metrics['sharpe_ratio']:.3f} → {meta_metrics['sharpe_ratio']:.3f} "
        f"({(meta_metrics['sharpe_ratio']/super_metrics['sharpe_ratio']-1)*100:+.1f}%)")

# Recommendation
print(f"\n3. Deployment Recommendation:")

sharpes = {
    'Best Base': best_base_metrics['sharpe_ratio'],
    'Enhanced Super': super_metrics['sharpe_ratio'],
    'Meta': meta_metrics['sharpe_ratio']
}

if qqq_metrics:
    sharpes['QQQ'] = qqq_metrics['sharpe_ratio']

best_strategy = max(sharpes, key=sharpes.get)
best_sharpe = sharpes[best_strategy]

print(f"   🏆 Best performer: {best_strategy} (Sharpe: {best_sharpe:.3f})")

if best_strategy == 'QQQ':
    print(f"   💡 Recommendation: Just buy and hold QQQ")
elif best_strategy == 'Best Base':
    print(f"   💡 Recommendation: Deploy sent_PPO")
elif best_strategy == 'Enhanced Super':
    print(f"   💡 Recommendation: Deploy Enhanced Super Agent")
else:
    print(f"   💡 Recommendation: Deploy Meta Agent")

# Save results
results = {
    'qqq_baseline': {k: float(v) if isinstance(v, (np.floating, float)) else v 
                    for k, v in qqq_metrics.items()} if qqq_metrics else None,
    'best_base_agent': {k: float(v) if isinstance(v, (np.floating, float)) else v 
                        for k, v in best_base_metrics.items()},
    'enhanced_super_agent': {
        'test_metrics': {k: float(v) if isinstance(v, (np.floating, float)) else v 
                        for k, v in super_metrics.items()},
        'val_sharpes': [float(s) for s in super_val_sharpes],
        'avg_val_sharpe': float(np.mean(super_val_sharpes)),
        'std_val_sharpe': float(np.std(super_val_sharpes))
    },
    'meta_agent': {
        'test_metrics': {k: float(v) if isinstance(v, (np.floating, float)) else v 
                        for k, v in meta_metrics.items()},
        'val_sharpes': [float(s) for s in meta_val_sharpes],
        'avg_val_sharpe': float(np.mean(meta_val_sharpes)),
        'std_val_sharpe': float(np.std(meta_val_sharpes))
    },
    'best_strategy': best_strategy,
    'enhancements': ['full_features', 'sac_algorithm', 'multi_window_ensemble']
}

os.makedirs('results', exist_ok=True)
with open('results/enhanced_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("\n" + "="*70)
print("FILES SAVED")
print("="*70)
print("\n✓ Models:")
print("  - models/super_agent_enhanced_ensemble/")
print("  - models/meta_agent_ensemble/")
print("\n✓ Results:")
print("  - results/enhanced_results.json")

print("\n" + "="*70)
print("TRAINING COMPLETE!")
print("="*70)
print(f"\n🎉 Deploy {best_strategy} for best performance!")


ENHANCED HIERARCHICAL SYSTEM

Enhancements:
  ✨ Super sees FULL features (base weights + technical + sentiment)
  ✨ Super uses SAC algorithm (better exploration)
  ✨ Multi-window ensemble (all windows, not just best)

Architecture:
  Base:  4 agents (tech_PPO, tech_SAC, sent_PPO, sent_SAC)
  Super: SAC with enhanced observations
  Meta:  PPO adjusting for regimes

[1/6] Loading data...
   ✓ QQQ benchmark loaded

   Trading data: (129, 10)
   Assets: ['NVDA', 'MU', 'MRVL', 'MSFT', 'ASML', 'AEM', 'AMD', 'GOOGL', 'PLUG', 'INGN']

[2/6] Creating walk-forward splits...

WALK-FORWARD VALIDATION SPLIT

Total data: 129 months
Date range: 2015-02 to 2025-10

Created 3 walk-forward windows:

  Window 1:
    Train: 2015-02 to 2020-01 (60 months)
    Val:   2020-02 to 2021-04 (15 months)

  Window 2:
    Train: 2015-02 to 2021-04 (75 months)
    Val:   2021-05 to 2022-07 (15 months)

  Window 3:
    Train: 2015-02 to 2022-07 (90 months)
    Val:   2022-08 to 2023-09 (14 months)

  Final Test:
    

NameError: name 'EnhancedSuperAgentEnv' is not defined